# CEPHA29 landmark model — train + export (Kaggle T4)

Trains a 29-landmark HRNet-W32 on CEPHA29 with **CephaloHRNet** (MIT) and exports a
web-compatible ONNX (FP32 + static QDQ INT8) for Cephalometry Studio.

**Kaggle setup**
1. Settings → Accelerator: **GPU T4 x2** (one is used).
2. Add Data → dataset **`felixtemko/cepha29`**.
3. Settings → Internet: **On** (for cloning + pretrained backbone).
4. Run all. Enable “Save Version” so outputs persist; resume from `last.pt` if a session ends.

Time: ~5–10 h at 768² with a pretrained backbone on a T4.

In [ ]:
!nvidia-smi
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# torch is preinstalled on Kaggle. Install the remaining deps.
!pip -q install timm albumentations opencv-python-headless pyyaml onnx onnxruntime pillow numpy scipy

In [ ]:
import os
REPO_URL = 'https://github.com/Muhammad-Nabeel-Sh/CephalometryStudio.git'  # for scripts/export-landmark-onnx.py
WORK = '/kaggle/working'
os.chdir(WORK)
%cd /kaggle/working
!rm -rf CephaloHRNet CephalometryStudio
!git clone --depth 1 https://github.com/Cestovatels/CephaloHRNet.git
!git clone --depth 1 $REPO_URL CephalometryStudio
print('cloned')

In [ ]:
import os
# Find the CEPHA29 root (a directory containing train/Cephalograms)
DATASET = None
for base, dirs, _ in os.walk('/kaggle/input'):
    if os.path.basename(base) == 'train' and 'Cephalograms' in dirs:
        DATASET = os.path.dirname(base)
        break
print('DATASET =', DATASET)
assert DATASET, 'CEPHA29 dataset not found — add the felixtemko/cepha29 dataset to this notebook.'

In [ ]:
# Train. imgsz is square; 768 matches the app's default. batch 4 + amp fits a T4 16GB.
# If you hit OOM, drop to --imgsz 512 --batch 4. If BN is unstable, keep batch >= 4.
!cd /kaggle/working/CephaloHRNet && python train.py \
  --data $DATASET \
  --model hrnet_w32 \
  --imgsz 768 \
  --batch 4 \
  --epochs 200 \
  --loss awing \
  --amp \
  --device 0 \
  --pretrained \
  --average_ann \
  --workers 2 \
  --patience 30 \
  --project runs/train \
  --name cepha29_w32_768
print('training done')

In [ ]:
import glob, os
best = '/kaggle/working/CephaloHRNet/runs/train/cepha29_w32_768/best.pt'
print('best.pt exists:', os.path.exists(best))
assert os.path.exists(best), 'best.pt not found — check the training output above.'
# A few real cephalograms for INT8 calibration + a verification overlay.
cal = sorted(glob.glob(os.path.join(DATASET, 'train', 'Cephalograms', '*')))[:16]
sample = cal[0]
print('calib images:', len(cal), '| sample:', os.path.basename(sample))
os.makedirs('/kaggle/working/calib', exist_ok=True)
import shutil
for f in cal:
    shutil.copy(f, '/kaggle/working/calib/' + os.path.basename(f))

In [ ]:
import os
# Export FP32 + web-compatible static QDQ INT8 ONNX.
SAMPLE = sorted(os.listdir('/kaggle/working/calib'))[0]
!cd /kaggle/working && python CephalometryStudio/scripts/export-landmark-onnx.py \
  --repo /kaggle/working/CephaloHRNet \
  --weights /kaggle/working/CephaloHRNet/runs/train/cepha29_w32_768/best.pt \
  --model hrnet_w32 --num-landmarks 29 \
  --input-size 768 \
  --out /kaggle/working/landmarks-cepha29.onnx \
  --quantize --calib-dir /kaggle/working/calib \
  --verify "/kaggle/working/calib/$SAMPLE"
print('\nOutputs in /kaggle/working:')
for f in os.listdir('/kaggle/working'):
    if f.endswith('.onnx') or f.endswith('-verify.png'):
        print(' ', f, round(os.path.getsize('/kaggle/working/'+f)/1e6,1), 'MB')

## Next
1. **Inspect** `landmarks-cepha29-verify.png` — every dot is labelled `index:symbol`; confirm it lands on the right anatomy (validates the 29-landmark channel order).
2. Download `landmarks-cepha29.int8.onnx` and its printed SHA-256.
3. Upload it to a GitHub Release / Hugging Face (CORS-enabled).
4. Update `src/data/landmarkModelInfo.js` (`landmarkSet: "cepha29"`, `inputSize: 768`, `url`, `sha256`) and build the CEPHA29 shape model (`python scripts/build-shape-model.py --set cepha29 --ann-dir <Annotations/.../Senior Orthodontists>`).